# LeetCode 2164 — Sort Even and Odd Indices Independently bằng Python

**Link đề:** https://leetcode.com/problems/sort-even-and-odd-indices-independently/
**Độ khó:** Easy · **Chủ đề:** Array, Sorting

```python
class Solution:
    def sortEvenOdd(self, nums: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Phân biệt **chỉ số chẵn/lẻ** (vị trí) với **giá trị chẵn/lẻ** — lỗi hiểu đề phổ biến nhất của bài.
2. Nắm slicing có bước `nums[::2]`, `nums[1::2]` và **gán vào slice** `nums[::2] = ...`.
3. Hiểu "non-decreasing" (tăng không ngặt) và "non-increasing" (giảm không ngặt); `sorted(..., reverse=True)`.
4. Đi từ cách tự sắp trong chỗ (selection sort theo bước 2) → tách–sắp–ghép → slicing một dòng → counting sort.
5. Trace và chứng minh vì sao hai nhóm **độc lập** với nhau.

## 1. Đề bài

Cho mảng `nums` (đánh chỉ số từ 0). Sắp xếp lại theo hai luật:
- Các giá trị ở **chỉ số lẻ** (1, 3, 5, ...) sắp **giảm dần không ngặt** (non-increasing).
- Các giá trị ở **chỉ số chẵn** (0, 2, 4, ...) sắp **tăng dần không ngặt** (non-decreasing).

Mỗi giá trị **giữ nguyên loại vị trí** của nó (chẵn ở lại chẵn, lẻ ở lại lẻ).

| `nums` | Kết quả | Giải thích |
|---|---|---|
| `[4,1,2,3]` | `[2,3,4,1]` | chẵn: `[4,2]` → `[2,4]`; lẻ: `[1,3]` → `[3,1]` |
| `[2,1]` | `[2,1]` | mỗi nhóm chỉ có 1 phần tử, không đổi |

**Constraints:** `1 <= len(nums) <= 100`, `1 <= nums[i] <= 100`.

In [1]:
nums = [4, 1, 2, 3]
for i, x in enumerate(nums):
    print(f"chỉ số {i} ({'chẵn' if i % 2 == 0 else 'lẻ '}) -> giá trị {x}")

chỉ số 0 (chẵn) -> giá trị 4
chỉ số 1 (lẻ ) -> giá trị 1
chỉ số 2 (chẵn) -> giá trị 2
chỉ số 3 (lẻ ) -> giá trị 3


## 2. Nền tảng Python cần biết

### 2.1. Chỉ số chẵn/lẻ: `i % 2`

`i % 2 == 0` ⟺ `i` chẵn. Đề nói về **vị trí** `i`, không phải giá trị `nums[i]`.

In [2]:
nums = [4, 1, 2, 3]
print("chỉ số chẵn:", [i for i in range(len(nums)) if i % 2 == 0])
print("giá trị ở chỉ số chẵn:", [nums[i] for i in range(len(nums)) if i % 2 == 0])
print("giá trị CHẴN (không phải điều đề hỏi):", [x for x in nums if x % 2 == 0])

chỉ số chẵn: [0, 2]
giá trị ở chỉ số chẵn: [4, 2]
giá trị CHẴN (không phải điều đề hỏi): [4, 2]


### 2.2. Slicing có bước: `nums[start::2]`

- `nums[::2]` = phần tử ở chỉ số `0, 2, 4, ...`.
- `nums[1::2]` = phần tử ở chỉ số `1, 3, 5, ...`.

In [3]:
nums = [10, 11, 12, 13, 14]
print("nums[::2]  =", nums[::2])
print("nums[1::2] =", nums[1::2])

nums[::2]  = [10, 12, 14]
nums[1::2] = [11, 13]


### 2.3. Gán vào slice: `nums[::2] = list_moi`

Python cho phép **ghi đè** các vị trí của một slice có bước bằng một list **cùng độ dài**. Đây là "vũ khí" của Cách 3.

In [4]:
nums = [10, 11, 12, 13, 14]
nums[::2] = [0, 0, 0]
print(nums)
try:
    nums[::2] = [1, 2]          # sai độ dài
except ValueError as e:
    print("ValueError:", e)

[0, 11, 0, 13, 0]
ValueError: attempt to assign sequence of size 2 to extended slice of size 3


### 2.4. Non-decreasing / non-increasing và `sorted(reverse=True)`

- **Non-decreasing** (tăng không ngặt): `a[0] <= a[1] <= ...` — cho phép bằng nhau. Đây là kết quả của `sorted(a)`.
- **Non-increasing** (giảm không ngặt): `a[0] >= a[1] >= ...` — `sorted(a, reverse=True)`.

In [5]:
a = [3, 1, 3, 2]
print("non-decreasing:", sorted(a))
print("non-increasing:", sorted(a, reverse=True))

non-decreasing: [1, 2, 3, 3]
non-increasing: [3, 3, 2, 1]


### 2.5. Ghép xen kẽ hai list

Nếu có `evens` (cho vị trí chẵn) và `odds` (cho vị trí lẻ), kết quả là `evens[0], odds[0], evens[1], odds[1], ...`. Phần tử ở chỉ số `i` lấy từ `evens[i // 2]` nếu `i` chẵn, từ `odds[i // 2]` nếu `i` lẻ.

In [6]:
evens, odds = [2, 4, 6], [9, 7]
merged = [evens[i // 2] if i % 2 == 0 else odds[i // 2] for i in range(len(evens) + len(odds))]
print(merged)
for i in range(5):
    print(i, "->", "evens" if i % 2 == 0 else "odds", "[", i // 2, "]")

[2, 9, 4, 7, 6]
0 -> evens [ 0 ]
1 -> odds [ 0 ]
2 -> evens [ 1 ]
3 -> odds [ 1 ]
4 -> evens [ 2 ]


Biểu thức `A if điều_kiện else B` là **biểu thức điều kiện** (ternary): trả `A` nếu điều kiện đúng, ngược lại `B`.

## 3. Cách 1 — Sắp trong chỗ bằng selection sort "nhảy 2 bước"

Cách người mới nghĩ khi chưa biết slicing: tự viết sắp xếp chọn (selection sort), nhưng chỉ so sánh các chỉ số **cùng tính chẵn/lẻ** (`j` chạy theo bước 2).

Selection sort: ở mỗi vị trí `i`, tìm phần tử "đúng" (nhỏ nhất hoặc lớn nhất) trong phần còn lại cùng nhóm rồi **hoán đổi** vào `i`.

In [7]:
def sort_even_odd_selection(nums: list[int]) -> list[int]:
    a = nums[:]                      # bản sao, không sửa input
    n = len(a)
    for i in range(n):
        best = i
        for j in range(i + 2, n, 2):  # chỉ cùng tính chẵn/lẻ với i
            if i % 2 == 0 and a[j] < a[best]:      # chẵn: lấy nhỏ nhất
                best = j
            if i % 2 == 1 and a[j] > a[best]:      # lẻ: lấy lớn nhất
                best = j
        a[i], a[best] = a[best], a[i]
    return a

print(sort_even_odd_selection([4, 1, 2, 3]))
print(sort_even_odd_selection([2, 1]))
print(sort_even_odd_selection([5, 39, 33, 5, 12, 27, 20]))

[2, 3, 4, 1]
[2, 1]
[5, 39, 12, 27, 20, 5, 33]


`a[i], a[best] = a[best], a[i]` là **hoán đổi** (swap) trong một dòng nhờ tuple unpacking.

### Chi phí cách 1

Hai vòng lồng nhau → `O(n²)`. Với `n ≤ 100` chạy tốt, nhưng code dài và dễ sai điều kiện. Nhận xét quan trọng: thực chất ta đang **sắp hai list riêng biệt** đan xen nhau.

## 4. Insight: hai nhóm độc lập → tách, sắp, ghép

Vị trí chẵn chỉ trao đổi giá trị với vị trí chẵn; vị trí lẻ với vị trí lẻ. Không có tương tác giữa hai nhóm ⇒ coi như **hai bài sắp xếp riêng**:

1. `evens = nums[::2]` → `sorted(evens)`.
2. `odds = nums[1::2]` → `sorted(odds, reverse=True)`.
3. Ghép xen kẽ trở lại.

Dùng `sorted` có sẵn (Timsort `O(k log k)`) thay cho selection sort tự viết.

In [8]:
nums = [5, 39, 33, 5, 12, 27, 20]
evens = sorted(nums[::2])
odds = sorted(nums[1::2], reverse=True)
print("evens (sắp tăng):", evens)
print("odds  (sắp giảm):", odds)

evens (sắp tăng): [5, 12, 20, 33]
odds  (sắp giảm): [39, 27, 5]


## 5. Cách 2 — Lời giải chính: tách – sắp – ghép

Từng đoạn:

**Đoạn 1 — tách:** `evens = nums[::2]`, `odds = nums[1::2]`.

**Đoạn 2 — sắp:** `evens.sort()`, `odds.sort(reverse=True)`.

**Đoạn 3 — ghép:** tạo `result = [0] * n`, rồi `result[i] = evens[i // 2]` nếu `i` chẵn, `odds[i // 2]` nếu lẻ.

In [9]:
def sort_even_odd_merge(nums: list[int]) -> list[int]:
    evens = nums[::2]                    # Đoạn 1
    odds = nums[1::2]
    evens.sort()                         # Đoạn 2
    odds.sort(reverse=True)
    result = [0] * len(nums)             # Đoạn 3
    for i in range(len(nums)):
        if i % 2 == 0:
            result[i] = evens[i // 2]
        else:
            result[i] = odds[i // 2]
    return result

print(sort_even_odd_merge([4, 1, 2, 3]))
print(sort_even_odd_merge([2, 1]))
print(sort_even_odd_merge([7]))

[2, 3, 4, 1]
[2, 1]
[7]


`[0] * n` tạo list `n` số 0 để sau đó ghi vào từng vị trí. Vì `nums[::2]` tạo **list mới** (bản sao), `evens.sort()` không ảnh hưởng `nums`.

## 6. TRACE từng bước

`nums = [5, 39, 33, 5, 12, 27, 20]` (n = 7).

- `evens = nums[::2] = [5, 33, 12, 20]` → sắp tăng `[5, 12, 20, 33]`.
- `odds = nums[1::2] = [39, 5, 27]` → sắp giảm `[39, 27, 5]`.

| `i` | chẵn/lẻ | lấy từ | `i // 2` | giá trị |
|:-:|:-:|---|:-:|:-:|
| 0 | chẵn | evens | 0 | 5 |
| 1 | lẻ | odds | 0 | 39 |
| 2 | chẵn | evens | 1 | 12 |
| 3 | lẻ | odds | 1 | 27 |
| 4 | chẵn | evens | 2 | 20 |
| 5 | lẻ | odds | 2 | 5 |
| 6 | chẵn | evens | 3 | 33 |

Kết quả `[5, 39, 12, 27, 20, 5, 33]`.

In [10]:
def trace_sort_even_odd(nums):
    evens = sorted(nums[::2])
    odds = sorted(nums[1::2], reverse=True)
    print("nums  =", nums)
    print("evens =", nums[::2], "-> sắp tăng", evens)
    print("odds  =", nums[1::2], "-> sắp giảm", odds)
    result = []
    for i in range(len(nums)):
        src, name = (evens, "evens") if i % 2 == 0 else (odds, "odds ")
        result.append(src[i // 2])
        print(f"i={i}: {name}[{i // 2}] = {src[i // 2]:>2} -> result={result}")
    return result

trace_sort_even_odd([5, 39, 33, 5, 12, 27, 20])

nums  = [5, 39, 33, 5, 12, 27, 20]
evens = [5, 33, 12, 20] -> sắp tăng [5, 12, 20, 33]
odds  = [39, 5, 27] -> sắp giảm [39, 27, 5]
i=0: evens[0] =  5 -> result=[5]
i=1: odds [0] = 39 -> result=[5, 39]
i=2: evens[1] = 12 -> result=[5, 39, 12]
i=3: odds [1] = 27 -> result=[5, 39, 12, 27]
i=4: evens[2] = 20 -> result=[5, 39, 12, 27, 20]
i=5: odds [2] =  5 -> result=[5, 39, 12, 27, 20, 5]
i=6: evens[3] = 33 -> result=[5, 39, 12, 27, 20, 5, 33]


[5, 39, 12, 27, 20, 5, 33]

## 7. Code nộp LeetCode — Cách 2

In [11]:
class Solution:
    def sortEvenOdd(self, nums: list[int]) -> list[int]:
        evens = sorted(nums[::2])
        odds = sorted(nums[1::2], reverse=True)
        result = []
        for i in range(len(nums)):
            result.append(evens[i // 2] if i % 2 == 0 else odds[i // 2])
        return result

sol = Solution()
print(sol.sortEvenOdd([4, 1, 2, 3]), sol.sortEvenOdd([2, 1]))

[2, 3, 4, 1] [2, 1]


## 8. Cách 3 — Gán vào slice (hai dòng)

Vì `nums[::2]` có đúng `len(sorted(nums[::2]))` phần tử, ta gán thẳng:

```python
nums[::2] = sorted(nums[::2])
nums[1::2] = sorted(nums[1::2], reverse=True)
```

Không cần bước ghép thủ công — Python lo việc đặt từng phần tử về đúng vị trí bước 2.

In [12]:
def sort_even_odd_slice(nums: list[int]) -> list[int]:
    a = nums[:]
    a[::2] = sorted(a[::2])
    a[1::2] = sorted(a[1::2], reverse=True)
    return a

print(sort_even_odd_slice([4, 1, 2, 3]), sort_even_odd_slice([5, 39, 33, 5, 12, 27, 20]))

[2, 3, 4, 1] [5, 39, 12, 27, 20, 5, 33]


In [13]:
class Solution:
    def sortEvenOdd(self, nums: list[int]) -> list[int]:
        nums[::2] = sorted(nums[::2])
        nums[1::2] = sorted(nums[1::2], reverse=True)
        return nums

print(Solution().sortEvenOdd([4, 1, 2, 3]))

[2, 3, 4, 1]


Bản nộp này **sửa trực tiếp** `nums` rồi trả lại chính nó — LeetCode chấp nhận. Trong code dùng lại, nên làm trên bản sao như `sort_even_odd_slice`.

### Thí nghiệm: sửa trong chỗ có ảnh hưởng biến bên ngoài không?

In [14]:
original = [4, 1, 2, 3]
out = Solution().sortEvenOdd(original)
print("out      =", out)
print("original =", original, "<- đã bị sửa!", out is original)

out      = [2, 3, 4, 1]
original = [2, 3, 4, 1] <- đã bị sửa! True


## 9. Cách 4 — Counting sort (giá trị ≤ 100)

Giá trị nằm trong `1..100` → đếm tần suất riêng cho nhóm chẵn và nhóm lẻ, rồi phát lại: nhóm chẵn đi từ giá trị **nhỏ đến lớn**, nhóm lẻ đi từ **lớn đến nhỏ**. Thời gian `O(n + V)` với `V = 100`.

In [15]:
def sort_even_odd_counting(nums: list[int]) -> list[int]:
    cnt_even = [0] * 101
    cnt_odd = [0] * 101
    for i, x in enumerate(nums):
        if i % 2 == 0:
            cnt_even[x] += 1
        else:
            cnt_odd[x] += 1
    evens = [v for v in range(1, 101) for _ in range(cnt_even[v])]
    odds = [v for v in range(100, 0, -1) for _ in range(cnt_odd[v])]
    result = nums[:]
    result[::2] = evens
    result[1::2] = odds
    return result

print(sort_even_odd_counting([4, 1, 2, 3]), sort_even_odd_counting([5, 39, 33, 5, 12, 27, 20]))

[2, 3, 4, 1] [5, 39, 12, 27, 20, 5, 33]


List comprehension **hai vòng** `[v for v in A for _ in range(cnt[v])]` đọc từ trái sang phải giống hai vòng `for` lồng nhau: với mỗi `v`, lặp `cnt[v]` lần, mỗi lần đưa `v` vào list.

In [16]:
cnt = {1: 2, 3: 1, 5: 3}
print([v for v in (1, 3, 5) for _ in range(cnt[v])])

[1, 1, 3, 5, 5, 5]


## 10. Vì sao đúng? (invariant)

- **Độc lập:** tập vị trí chẵn `E = {0, 2, 4, ...}` và lẻ `O = {1, 3, ...}` rời nhau và phủ kín `0..n-1`. Đề yêu cầu sắp các giá trị **trong E** và **trong O** riêng → kết quả trên `E` chỉ phụ thuộc vào multiset giá trị ở `E`, tương tự `O`.
- **Đúng thứ tự trên E:** phần tử thứ `t` của `E` theo thứ tự tăng là vị trí `2t`; ta đặt `evens_sorted[t]` vào `2t` ⇒ dãy trên `E` không giảm.
- **Đúng thứ tự trên O:** vị trí `2t + 1` nhận `odds_sorted_desc[t]` ⇒ dãy trên `O` không tăng.
- **Bảo toàn phần tử:** sắp xếp chỉ hoán vị, không thêm/bớt → kết quả là hoán vị của `nums`.
- Thứ tự hai bước "sắp lẻ trước, chẵn sau" trong đề **không quan trọng** vì hai bước không đụng vào nhau.

**Selection sort (Cách 1)**: invariant sau vòng `i`, vị trí `i` chứa giá trị nhỏ nhất (nếu chẵn) / lớn nhất (nếu lẻ) trong các vị trí cùng nhóm `≥ i`.

## 11. Độ phức tạp

Gọi `n = len(nums)`, `V = 100`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Selection sort bước 2 | `O(n²)` | `O(n)` (bản sao) hoặc `O(1)` thêm nếu sửa tại chỗ | không cần hàm có sẵn |
| 2. Tách – sắp – ghép | `O(n log n)` | `O(n)` | lời giải chính, rõ ràng |
| 3. Gán vào slice | `O(n log n)` | `O(n)` (slice tạo bản sao) | gọn nhất |
| 4. Counting | `O(n + V)` | `O(V + n)` | tận dụng giá trị nhỏ |

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

1. Hai ví dụ của đề.
2. `n = 1`, `n = 2`, `n = 3` (lẻ phần tử: nhóm chẵn dài hơn).
3. Giá trị trùng nhau.
4. Mảng đã sắp sẵn / sắp ngược.
5. Đối chiếu ngẫu nhiên + kiểm tra **tính chất**: nhóm chẵn không giảm, nhóm lẻ không tăng, kết quả là hoán vị của input.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS {actual}")

impls = [sort_even_odd_selection, sort_even_odd_merge, sort_even_odd_slice, sort_even_odd_counting,
         lambda a: Solution().sortEvenOdd(a[:])]
cases = [
    ("đề ví dụ 1", [4, 1, 2, 3], [2, 3, 4, 1]),
    ("đề ví dụ 2", [2, 1], [2, 1]),
    ("n = 1", [7], [7]),
    ("n = 3", [3, 1, 2], [2, 1, 3]),
    ("trùng giá trị", [5, 5, 5, 5], [5, 5, 5, 5]),
    ("đã sắp tăng", [1, 2, 3, 4, 5, 6], [1, 6, 3, 4, 5, 2]),
    ("dài 7", [5, 39, 33, 5, 12, 27, 20], [5, 39, 12, 27, 20, 5, 33]),
]
for name, nums, exp in cases:
    for f in impls:
        assert f(list(nums)) == exp, (name, f)
    check(name, sort_even_odd_merge(nums), exp)

đề ví dụ 1         PASS [2, 3, 4, 1]
đề ví dụ 2         PASS [2, 1]
n = 1              PASS [7]
n = 3              PASS [2, 1, 3]
trùng giá trị      PASS [5, 5, 5, 5]
đã sắp tăng        PASS [1, 6, 3, 4, 5, 2]
dài 7              PASS [5, 39, 12, 27, 20, 5, 33]


In [18]:
import random
from collections import Counter
rng = random.Random(2164)
for _ in range(500):
    nums = [rng.randint(1, 100) for _ in range(rng.randint(1, 15))]
    out = sort_even_odd_merge(nums)
    ev, od = out[::2], out[1::2]
    assert all(ev[i] <= ev[i + 1] for i in range(len(ev) - 1))      # chẵn không giảm
    assert all(od[i] >= od[i + 1] for i in range(len(od) - 1))      # lẻ không tăng
    assert Counter(ev) == Counter(nums[::2]) and Counter(od) == Counter(nums[1::2])
    for f in impls:
        assert f(list(nums)) == out
print("500 test ngẫu nhiên + kiểm tra tính chất: PASS")

500 test ngẫu nhiên + kiểm tra tính chất: PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Hiểu nhầm thành giá trị chẵn/lẻ

In [19]:
def wrong_by_value(nums):
    ev = sorted(x for x in nums if x % 2 == 0)
    od = sorted((x for x in nums if x % 2 == 1), reverse=True)
    return ev + od

print("sai :", wrong_by_value([4, 1, 2, 3]), "| đúng:", sort_even_odd_merge([4, 1, 2, 3]))

sai : [2, 4, 3, 1] | đúng: [2, 3, 4, 1]


**Vì sao sai:** đề nói "**indices**" (chỉ số), không phải "values". **Sửa:** phân nhóm theo `i % 2`, giữ nguyên vị trí loại chẵn/lẻ.

### Lỗi 2: Đảo chiều sắp (chẵn giảm, lẻ tăng)

In [20]:
def wrong_direction(nums):
    a = nums[:]
    a[::2] = sorted(a[::2], reverse=True)
    a[1::2] = sorted(a[1::2])
    return a

print("sai :", wrong_direction([4, 1, 2, 3]), "| đúng:", sort_even_odd_slice([4, 1, 2, 3]))

sai : [4, 1, 2, 3] | đúng: [2, 3, 4, 1]


**Vì sao sai:** đọc nhầm "non-increasing" (giảm) với "non-decreasing" (tăng). Mẹo nhớ: "non-**de**creasing" = "không giảm" = tăng dần. **Sửa:** chẵn → `sorted(...)`, lẻ → `sorted(..., reverse=True)`.

### Lỗi 3: Dùng `.sort()` trên slice, tưởng là sắp luôn `nums`

In [21]:
nums = [4, 1, 2, 3]
nums[::2].sort()           # sắp một BẢN SAO rồi vứt đi
print(nums, "<- không đổi gì")

[4, 1, 2, 3] <- không đổi gì


**Vì sao sai:** `nums[::2]` tạo **list mới**; `.sort()` sắp list mới đó, `nums` không đổi. **Sửa:** `nums[::2] = sorted(nums[::2])`.

### Lỗi 4: Ghép bằng `zip` khi số phần tử lẻ → mất phần tử cuối

In [22]:
evens, odds = [5, 12, 20, 33], [39, 27, 5]
wrong = [x for pair in zip(evens, odds) for x in pair]
print("zip ghép :", wrong, "<- mất 33!")
right = [x for pair in zip(evens, odds) for x in pair] + evens[len(odds):]
print("đã sửa   :", right)

zip ghép : [5, 39, 12, 27, 20, 5] <- mất 33!
đã sửa   : [5, 39, 12, 27, 20, 5, 33]


**Vì sao sai:** `zip` dừng ở list **ngắn hơn**. Khi `n` lẻ, `evens` dài hơn `odds` một phần tử. **Sửa:** nối thêm phần dư, hoặc dùng `itertools.zip_longest`, hoặc ghép theo chỉ số `i // 2`.

## 14. Mẹo nhận dạng pattern

- "Chỉ số chẵn / lẻ", "mỗi phần tử thứ k" → slicing bước `k`: `a[r::k]`.
- "Sắp **độc lập** từng nhóm vị trí" → tách nhóm, sắp, đặt lại; trong Python dùng **gán vào slice**.
- Đọc kỹ "non-decreasing" = tăng (cho phép bằng), "non-increasing" = giảm.
- Giá trị nhỏ (≤ 100) → counting sort là lựa chọn `O(n + V)`.

Bài liên quan: #905 Sort Array By Parity, #922 Sort Array By Parity II, #2161 Partition Array According to Given Pivot, #1630 Arithmetic Subarrays.

## 15. Biến thể

### 15.1. Sắp độc lập theo **k nhóm** chỉ số (`i % k`)

In [23]:
def sort_by_residue(nums, k, descending_groups=()):
    a = nums[:]
    for r in range(k):
        a[r::k] = sorted(a[r::k], reverse=(r in descending_groups))
    return a

print(sort_by_residue([9, 8, 7, 6, 5, 4, 3, 2, 1], 3))
print(sort_by_residue([4, 1, 2, 3], 2, descending_groups={1}), "<- chính là bài gốc")

[3, 2, 1, 6, 5, 4, 9, 8, 7]
[2, 3, 4, 1] <- chính là bài gốc


### 15.2. LeetCode 922 — Sort Array By Parity II
Sắp lại sao cho giá trị chẵn nằm ở chỉ số chẵn, giá trị lẻ ở chỉ số lẻ (biết số lượng bằng nhau). Ở đây "giá trị" và "chỉ số" cùng xuất hiện — rất tốt để luyện phân biệt.

In [24]:
def sort_array_by_parity_ii(nums):
    ev = [x for x in nums if x % 2 == 0]
    od = [x for x in nums if x % 2 == 1]
    res = [0] * len(nums)
    res[::2] = ev
    res[1::2] = od
    return res

print(sort_array_by_parity_ii([4, 2, 5, 7]))

[4, 5, 2, 7]


## 16. Bài tập (TODO — tự làm, đáp án ở cuối)

### Bài 1 — Đoán output
1. `[9, 8, 7, 6, 5][1::2]` = ?
2. `sortEvenOdd([1, 2, 3, 4, 5])` = ?
3. Sau `a = [3, 1, 2]; a[::2] = [0, 0]`, `a` = ?

### Bài 2 — Trace tay
Lập bảng như mục TRACE cho `nums = [6, 1, 4, 9, 2]`.

### Bài 3 — Điền chỗ trống (Cách 3)

In [25]:
def exercise_sort_even_odd(nums: list[int]) -> list[int]:
    a = nums[:]
    # TODO 1: gán a[::2] bằng các phần tử chỉ số chẵn đã sắp tăng
    # TODO 2: gán a[1::2] bằng các phần tử chỉ số lẻ đã sắp giảm
    return a

# assert exercise_sort_even_odd([4, 1, 2, 3]) == [2, 3, 4, 1]

### Bài 4 — Viết lại **không dùng slicing** và không dùng `sorted`
Dùng vòng lặp tách nhóm, tự viết insertion sort (hoặc selection sort) cho từng nhóm, rồi ghép.

In [26]:
def exercise_no_slicing(nums: list[int]) -> list[int]:
    # TODO
    pass

# assert exercise_no_slicing([4, 1, 2, 3]) == [2, 3, 4, 1]
# assert exercise_no_slicing([5, 39, 33, 5, 12, 27, 20]) == [5, 39, 12, 27, 20, 5, 33]

### Bài 5 — Biến thể: cả hai nhóm sắp **tăng dần**, sau đó trả về `True` nếu toàn mảng đã tăng dần không ngặt

In [27]:
def exercise_both_ascending_is_sorted(nums: list[int]) -> bool:
    # TODO
    pass

# assert exercise_both_ascending_is_sorted([3, 4, 1, 2]) == True     # -> [1, 2, 3, 4]
# assert exercise_both_ascending_is_sorted([1, 4, 2, 3]) == False    # -> [1, 3, 2, 4]

### Bài 6 — Mini challenge: sắp tại chỗ, `O(1)` bộ nhớ thêm
Viết hàm sửa trực tiếp `nums` (không tạo list mới nào có độ dài phụ thuộc `n`) để ra kết quả bài gốc.

In [28]:
def exercise_in_place(nums: list[int]) -> None:
    # TODO: insertion sort với bước nhảy 2, điều kiện so sánh phụ thuộc i % 2
    pass

# a = [5, 39, 33, 5, 12, 27, 20]; exercise_in_place(a); assert a == [5, 39, 12, 27, 20, 5, 33]

### Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Bước 2 bắt đầu từ chỉ số 1 | chẵn `[1,3,5]`, lẻ `[2,4]` | lẻ sắp giảm `[4,2]` |
| 2 | Tách trước | evens `[6,4,2]`, odds `[1,9]` | `[2,9,4,1,6]` |
| 3 | `sorted` trả list mới | `a[::2] = sorted(a[::2])` | `reverse=True` cho lẻ |
| 4 | Hai list `ev`, `od` | insertion sort: đẩy phần tử về trái khi còn sai thứ tự | ghép theo `i // 2` |
| 5 | Sắp cả hai tăng | kiểm `all(a[i] <= a[i+1] ...)` | dùng Cách 3 với `reverse=False` |
| 6 | Insertion sort với `j -= 2` | hàm `wrong_order(x, y, i)` phụ thuộc nhóm | `while j >= 0 and sai_thu_tu: nums[j+2] = nums[j]; j -= 2` |

## 17. Đáp án tham khảo (tự làm trước rồi hãy xem)

**Bài 1:** `[8, 6]`; `[1, 4, 3, 2, 5]`; `[0, 1, 0]`.

**Bài 2:** evens `[6,4,2]` → `[2,4,6]`; odds `[1,9]` → `[9,1]`; kết quả `[2, 9, 4, 1, 6]`.

In [29]:
a = [3, 1, 2]; a[::2] = [0, 0]
print([9, 8, 7, 6, 5][1::2], Solution().sortEvenOdd([1, 2, 3, 4, 5]), a)
trace_sort_even_odd([6, 1, 4, 9, 2])

[8, 6] [1, 4, 3, 2, 5] [0, 1, 0]
nums  = [6, 1, 4, 9, 2]
evens = [6, 4, 2] -> sắp tăng [2, 4, 6]
odds  = [1, 9] -> sắp giảm [9, 1]
i=0: evens[0] =  2 -> result=[2]
i=1: odds [0] =  9 -> result=[2, 9]
i=2: evens[1] =  4 -> result=[2, 9, 4]
i=3: odds [1] =  1 -> result=[2, 9, 4, 1]
i=4: evens[2] =  6 -> result=[2, 9, 4, 1, 6]


[2, 9, 4, 1, 6]

In [30]:
def answer_sort_even_odd(nums):
    a = nums[:]
    a[::2] = sorted(a[::2])
    a[1::2] = sorted(a[1::2], reverse=True)
    return a


def _insertion_sort(lst, descending=False):
    lst = lst[:]
    for i in range(1, len(lst)):
        key = lst[i]
        j = i - 1
        while j >= 0 and ((lst[j] < key) if descending else (lst[j] > key)):
            lst[j + 1] = lst[j]
            j -= 1
        lst[j + 1] = key
    return lst


def answer_no_slicing(nums):
    ev, od = [], []
    for i in range(len(nums)):
        (ev if i % 2 == 0 else od).append(nums[i])
    ev = _insertion_sort(ev)
    od = _insertion_sort(od, descending=True)
    res = []
    for i in range(len(nums)):
        res.append(ev[i // 2] if i % 2 == 0 else od[i // 2])
    return res


def answer_both_ascending_is_sorted(nums):
    a = nums[:]
    a[::2] = sorted(a[::2])
    a[1::2] = sorted(a[1::2])
    return all(a[i] <= a[i + 1] for i in range(len(a) - 1))


def answer_in_place(nums):
    n = len(nums)
    for i in range(2, n):
        key = nums[i]
        j = i - 2
        if i % 2 == 0:
            while j >= 0 and nums[j] > key:     # chẵn: tăng dần
                nums[j + 2] = nums[j]
                j -= 2
        else:
            while j >= 0 and nums[j] < key:     # lẻ: giảm dần
                nums[j + 2] = nums[j]
                j -= 2
        nums[j + 2] = key


assert answer_sort_even_odd([4, 1, 2, 3]) == [2, 3, 4, 1]
assert answer_no_slicing([4, 1, 2, 3]) == [2, 3, 4, 1]
assert answer_no_slicing([5, 39, 33, 5, 12, 27, 20]) == [5, 39, 12, 27, 20, 5, 33]
assert answer_both_ascending_is_sorted([3, 4, 1, 2]) == True
assert answer_both_ascending_is_sorted([1, 4, 2, 3]) == False
a = [5, 39, 33, 5, 12, 27, 20]; answer_in_place(a); assert a == [5, 39, 12, 27, 20, 5, 33]
for _ in range(300):
    nums = [rng.randint(1, 100) for _ in range(rng.randint(1, 15))]
    b2 = nums[:]; answer_in_place(b2)
    assert b2 == answer_no_slicing(nums) == sort_even_odd_merge(nums)
print("Đáp án tham khảo: PASS toàn bộ test!")

Đáp án tham khảo: PASS toàn bộ test!


Mẹo trong `answer_no_slicing`: `(ev if i % 2 == 0 else od).append(...)` chọn **list đích** bằng biểu thức điều kiện rồi gọi `append` lên nó.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def sortEvenOdd(self, nums: list[int]) -> list[int]:
        nums[::2] = sorted(nums[::2])                    # chỉ số chẵn: tăng
        nums[1::2] = sorted(nums[1::2], reverse=True)    # chỉ số lẻ: giảm
        return nums
```

**Khung tư duy:** nhóm vị trí độc lập → tách bằng slice bước `k` → sắp từng nhóm theo chiều yêu cầu → gán lại vào slice.

**Checklist trước khi nộp:**
- [ ] Phân nhóm theo **chỉ số**, không theo giá trị?
- [ ] Chẵn **tăng**, lẻ **giảm** (đúng chiều)?
- [ ] Không gọi `.sort()` trên slice (bản sao)?
- [ ] `n` lẻ: nhóm chẵn dài hơn — ghép không mất phần tử cuối?
- [ ] Test `n = 1`, `n = 2`, giá trị trùng.